## Подготовка данных (для всех задач)

In [3]:
import pandas as pd
import numpy as np

data = {
    'date': pd.date_range('2023-01-01', periods=100),
    'city': np.random.choice(['Москва', 'СПб', 'Екатеринбург'], 100),
    'category': np.random.choice(['Электроника', 'Одежда', 'Продукты'], 100),
    'product_id': np.random.randint(1000, 2000, 100),
    'sales': np.random.randint(1000, 30000, 100),
    'quantity': np.random.randint(1, 10, 100),
    'price': np.random.randint(500, 5000, 100)
}

df = pd.DataFrame(data)
df.to_csv('sales.csv', index=False)

print("Файл sales.csv создан, размер:", df.shape)
df.head()

Файл sales.csv создан, размер: (100, 7)


,date,city,category,product_id,sales,quantity,price
0,2023-01-01,Екатеринбург,Одежда,1488,20567,8,4859
1,2023-01-02,Москва,Продукты,1918,6770,2,4057
2,2023-01-03,Москва,Электроника,1317,17288,2,1957
3,2023-01-04,Москва,Электроника,1104,23954,7,1813
4,2023-01-05,Москва,Одежда,1887,12706,7,3425


## Задача 1. Чтение и первичная обработка

In [4]:
# 1) Загрузка
df = pd.read_csv('sales.csv')

# 2) Первые 5 строк
print("=== Первые 5 строк ===")
print(df.head())

# 3) Проверка пропущенных значений
print("\n=== Пропущенные значения по столбцам ===")
print(df.isna().sum())

# 4) Основная статистика по числовым столбцам
print("\n=== describe() ===")
print(df.describe())

=== Первые 5 строк ===
         date          city     category  product_id  sales  quantity  price
0  2023-01-01  Екатеринбург       Одежда        1488  20567         8   4859
1  2023-01-02        Москва     Продукты        1918   6770         2   4057
2  2023-01-03        Москва  Электроника        1317  17288         2   1957
3  2023-01-04        Москва  Электроника        1104  23954         7   1813
4  2023-01-05        Москва       Одежда        1887  12706         7   3425

=== Пропущенные значения по столбцам ===
date          0
city          0
category      0
product_id    0
sales         0
quantity      0
price         0
dtype: int64

=== describe() ===
        product_id         sales    quantity        price
count   100.000000    100.000000  100.000000   100.000000
mean   1521.680000  15896.750000    5.300000  2884.500000
std     291.538132   7663.364065    2.687419  1243.676186
min    1013.000000   1150.000000    1.000000   586.000000
25%    1281.250000   9328.500000    3.

## Задача 2. Фильтрация данных

In [5]:
df = pd.read_csv('sales.csv', parse_dates=['date'])

# 2.1. Продажи больше 1000
big_sales = df[df['sales'] > 1000]
print("Записей с sales > 1000:", len(big_sales))
print(big_sales.head())

# 2.2. Записи за определённый период (март 2023)
march = df[(df['date'] >= '2023-03-01') & (df['date'] <= '2023-03-31')]
print("\nЗаписей за март 2023:", len(march))
print(march.head())

# 2.3. Несколько условий одновременно
# Москва ИЛИ СПб, категория "Электроника", продажи > 5000
multi = df[
    (df['city'].isin(['Москва', 'СПб'])) &
    (df['category'] == 'Электроника') &
    (df['sales'] > 5000)
]
print("\nПо нескольким условиям найдено записей:", len(multi))
print(multi.head())

Записей с sales > 1000: 100
        date          city     category  product_id  sales  quantity  price
0 2023-01-01  Екатеринбург       Одежда        1488  20567         8   4859
1 2023-01-02        Москва     Продукты        1918   6770         2   4057
2 2023-01-03        Москва  Электроника        1317  17288         2   1957
3 2023-01-04        Москва  Электроника        1104  23954         7   1813
4 2023-01-05        Москва       Одежда        1887  12706         7   3425

Записей за март 2023: 31
         date          city     category  product_id  sales  quantity  price
59 2023-03-01  Екатеринбург  Электроника        1368  12580         8   2957
60 2023-03-02        Москва  Электроника        1982   7281         8    786
61 2023-03-03  Екатеринбург  Электроника        1121  15423         2   3979
62 2023-03-04  Екатеринбург  Электроника        1859  14687         6   4786
63 2023-03-05  Екатеринбург     Продукты        1950   8447         1   2855

По нескольким условиям найд

## Задача 3. Группировка данных

In [6]:
df = pd.read_csv('sales.csv')

# 3.1. Суммарные продажи по городам
sales_by_city = df.groupby('city')['sales'].sum().sort_values(ascending=False)
print("=== Суммарные продажи по городам ===")
print(sales_by_city)

# 3.2. Средний чек по категориям
# Чек = sales / quantity, т.е средняя стоимость одной покупки в категории
df['avg_check'] = df['sales'] / df['quantity']
avg_check_by_category = df.groupby('category')['avg_check'].mean().round(2)
print("\n=== Средний чек по категориям ===")
print(avg_check_by_category)

# 3.3. Топ-5 по объёму покупок
# Объём покупок — суммарное quantity
top5 = (df.groupby('product_id')['quantity']
          .sum()
          .sort_values(ascending=False)
          .head(5))
print("\n=== Топ-5 товаров по количеству проданных единиц ===")
print(top5)

=== Суммарные продажи по городам ===
city
Москва          655724
Екатеринбург    522767
СПб             411184
Name: sales, dtype: int64

=== Средний чек по категориям ===
category
Одежда         4941.11
Продукты       5220.78
Электроника    4502.10
Name: avg_check, dtype: float64

=== Топ-5 товаров по количеству проданных единиц ===
product_id
1441    13
1399    11
1708    11
1950    10
1013     9
Name: quantity, dtype: int64


## Задача 4. Работа с NaN

In [7]:
df = pd.read_csv('sales.csv')

# Вносим NaN в пару столбцов для демонстрации
df.loc[df.sample(10, random_state=42).index, 'sales'] = np.nan
df.loc[df.sample(7, random_state=1).index, 'price'] = np.nan

print("Пропуски до обработки:")
print(df.isna().sum())

Пропуски до обработки:
date           0
city           0
category       0
product_id     0
sales         10
quantity       0
price          7
dtype: int64


4.1 Заполнить пропуски средним значением

In [8]:
df_filled = df.copy()

df_filled['sales'] = df_filled['sales'].fillna(df_filled['sales'].mean())
df_filled['price'] = df_filled['price'].fillna(df_filled['price'].mean())

print("Пропуски после fillna:")
print(df_filled.isna().sum())

Пропуски после fillna:
date          0
city          0
category      0
product_id    0
sales         0
quantity      0
price         0
dtype: int64


4.2. Удалить строки с пропусками

In [9]:
df_dropped = df.dropna()                   # удаляет строки с NaN хотя бы в одном столбце
print("Размер до:", df.shape, "| после:", df_dropped.shape)

# Или только по конкретным столбцам
df_dropped_subset = df.dropna(subset=['sales', 'price'])
print("После dropna по подмножеству:", df_dropped_subset.shape)

Размер до: (100, 7) | после: (84, 7)
После dropna по подмножеству: (84, 7)


4.3. Создать новый столбец на основе существующих

In [10]:
# Вариант А: цена со скидкой 10%
df['price_discount'] = df['price'] * 0.9

# Вариант Б: доход = количество × цена за единицу
df['revenue'] = df['quantity'] * (df['sales'] / df['quantity'])  # = просто sales, но наглядно

# Вариант В: категория чека (маленький / средний / крупный)
df['check_type'] = pd.cut(
    df['sales'],
    bins=[0, 5000, 15000, float('inf')],
    labels=['мелкий', 'средний', 'крупный']
)

print(df[['sales', 'price', 'price_discount', 'check_type']].head(10))
print("\nРаспределение по типу чека:")
print(df['check_type'].value_counts())

     sales   price  price_discount check_type
0      NaN  4859.0          4373.1        NaN
1   6770.0  4057.0          3651.3    средний
2  17288.0  1957.0          1761.3    крупный
3  23954.0  1813.0          1631.7    крупный
4  12706.0  3425.0          3082.5    средний
5   5388.0  4427.0          3984.3    средний
6  13159.0  3908.0          3517.2    средний
7  28566.0  3298.0          2968.2    крупный
8  23516.0  4113.0          3701.7    крупный
9  27265.0  4696.0          4226.4    крупный

Распределение по типу чека:
check_type
крупный    46
средний    38
мелкий      6
Name: count, dtype: int64
